# 02 - Pipeline de Machine Learning

**Datathon Educação — Passos Mágicos**

Este notebook demonstra o pipeline completo de Machine Learning, desde o pré-processamento até o treinamento e serialização do modelo.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.utils import generate_synthetic_data, load_raw_data, YEARS, NUMERIC_INDICATORS

print('Setup completo.')

## 1. Carregamento dos Dados

In [ ]:
try:
    df = load_raw_data()
    data_source = 'reais'
except FileNotFoundError:
    df = generate_synthetic_data(n_samples=500)
    data_source = 'sinteticos'

print(f'Dados {data_source} carregados: {df.shape}')
df.head(3)

## 2. Pré-processamento

### 2.1 Remoção de Identificadores

Colunas como NOME, ID, MATRICULA são removidas pois não contribuem para o modelo.

In [ ]:
from src.preprocessing import drop_identifier_columns, handle_missing_values

print(f'Colunas antes: {len(df.columns)}')
df = drop_identifier_columns(df)
print(f'Colunas apos remocao de identificadores: {len(df.columns)}')

### 2.2 Tratamento de Valores Ausentes

Estratégia: **mediana** para colunas numéricas, **moda** para categóricas.

In [ ]:
print(f'Nulos antes: {df.isnull().sum().sum()}')
df = handle_missing_values(df, strategy='median')
print(f'Nulos apos tratamento: {df.isnull().sum().sum()}')

## 3. Feature Engineering

Criamos três tipos de features adicionais:

1. **Temporais**: diferenças ano-a-ano, tendência, média e desvio padrão
2. **Compostas**: Academic Composite, Engagement Composite, Risk Score
3. **Interação**: INDE×IEG, IPS×IAA, Bolsista×INDE

In [ ]:
from src.feature_engineering import feature_engineering_pipeline

n_before = len(df.columns)
df = feature_engineering_pipeline(df)
n_after = len(df.columns)

print(f'Features antes: {n_before}')
print(f'Features depois: {n_after} (+{n_after - n_before} novas)')

new_cols = [c for c in df.columns if any(x in c for x in ['_diff_', '_trend', '_mean', '_std', 'COMPOSITE', 'RISK_SCORE', 'INTERACTION', 'PROGRESS', 'GAP'])]
print(f'\nFeatures criadas: {new_cols[:15]}...')

## 4. Extração do Target e Encoding

In [ ]:
from src.preprocessing import extract_target, encode_categorical_columns, split_data, normalize_features

X, y = extract_target(df)
print(f'Features: {X.shape}')
print(f'Target: {y.shape}')
print(f'Distribuicao: 0={int((y==0).sum())} ({(y==0).mean():.1%}), 1={int((y==1).sum())} ({(y==1).mean():.1%})')

In [ ]:
X, encoders = encode_categorical_columns(X)
print(f'Encoders criados: {list(encoders.keys())}')
print(f'Shape apos encoding: {X.shape}')

## 5. Split e Normalização

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train target: 0={int((y_train==0).sum())}, 1={int((y_train==1).sum())}')
print(f'Test target:  0={int((y_test==0).sum())}, 1={int((y_test==1).sum())}')

In [ ]:
X_train, X_test, scaler = normalize_features(X_train, X_test)
print(f'Normalizacao aplicada. Medias do train (primeiras 5): {X_train.iloc[:, :5].mean().values.round(3)}')

## 6. Cross-Validation de Modelos

Avaliamos 5 modelos com cross-validation estratificada (5 folds) usando F1-Score.

In [ ]:
from src.train import cross_validate_models

cv_results = cross_validate_models(X_train, y_train, scoring='f1')
cv_results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette('viridis', len(cv_results))
bars = ax.barh(cv_results['model'], cv_results['mean_score'], 
               xerr=cv_results['std_score'], color=colors, alpha=0.8)
ax.set_xlabel('F1-Score (CV)')
ax.set_title('Comparacao de Modelos — Cross-Validation')
for bar, score in zip(bars, cv_results['mean_score']):
    ax.text(score + 0.005, bar.get_y() + bar.get_height()/2, 
            f'{score:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

## 7. Treinamento com Tuning de Hiperparâmetros

O melhor modelo é selecionado automaticamente e refinado com GridSearchCV.

In [ ]:
from src.train import train_best_model, save_model, save_pipeline_artifacts

best_model, _, best_params = train_best_model(X_train, y_train, scoring='f1')

print(f'\nModelo selecionado: {type(best_model).__name__}')
print(f'Melhores hiperparametros: {best_params}')

## 8. Salvamento dos Artefatos

In [ ]:
model_path = save_model(best_model)
print(f'Modelo salvo em: {model_path}')

artifacts = {
    'encoders': encoders,
    'scaler': scaler,
    'feature_names': X_train.columns.tolist(),
}
pipeline_path = save_pipeline_artifacts(artifacts)
print(f'Pipeline salvo em: {pipeline_path}')

## 9. Resumo do Pipeline

| Etapa | Descrição | Resultado |
|-------|-----------|----------|
| Carregamento | Dados PEDE 2020-2022 | Dataset com indicadores por ano |
| Pré-processamento | Drop IDs, imputação, encoding | Dados limpos e numéricos |
| Feature Engineering | Temporal, compostas, interação | +N novas features |
| Split | 80/20 estratificado | Train e test balanceados |
| Normalização | StandardScaler | Features padronizadas |
| Cross-Validation | 5 modelos, 5 folds | Ranking por F1-Score |
| Tuning | GridSearchCV no melhor modelo | Hiperparâmetros otimizados |
| Serialização | joblib | model.joblib + pipeline.joblib |

---

**Próximo passo**: Notebook `03_Avaliacao_Modelo.ipynb` — Avaliação detalhada